In [4]:
import chromadb
print("ChromaDB imported successfully!")
print(chromadb.__version__)

ChromaDB imported successfully!
1.5.9


In [5]:
import chromadb
import os

# Create a ChromaDB client (saves to disk, so it persists between sessions)
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Create a collection (like a "table" for our documents)
collection = chroma_client.get_or_create_collection(name="lending_policies")

# Load all our policy documents
policy_folder = "policy_docs"
documents = []
doc_ids = []

for filename in os.listdir(policy_folder):
    filepath = os.path.join(policy_folder, filename)
    with open(filepath, 'r') as f:
        content = f.read()
        documents.append(content)
        doc_ids.append(filename.replace('.txt', ''))

print(f"Loaded {len(documents)} documents:")
print(doc_ids)

Loaded 6 documents:
['adverse_action_requirements', 'credit_grade', 'credit_history', 'debt_to_income', 'income_verification', 'loan_terms']


In [6]:
collection.add(
    documents=documents,
    ids=doc_ids
)

print("Documents successfully embedded and indexed!")
print("Total documents in collection:", collection.count())

Documents successfully embedded and indexed!
Total documents in collection: 6


In [7]:
# Test query: imagine SHAP told us "term" and "sub_grade" were top reasons
test_query = "loan term length and credit sub-grade risk"

results = collection.query(
    query_texts=[test_query],
    n_results=2  # return the top 2 most relevant documents
)

print("Query:", test_query)
print("\nTop matching documents:")
for doc_id, doc_text in zip(results['ids'][0], results['documents'][0]):
    print(f"\n--- {doc_id} ---")
    print(doc_text[:200], "...")  # print first 200 characters as a preview

Query: loan term length and credit sub-grade risk

Top matching documents:

--- loan_terms ---
Loan Term and Amount Policy

Loan term length (e.g., 36 months vs. 60 months) directly affects risk assessment. 
Longer loan terms carry greater uncertainty over the repayment period, including 
expos ...

--- credit_grade ---
Credit Grade and Sub-Grade Policy

Loan applicants are assigned a credit grade (A through G) and a more granular 
sub-grade (e.g., A1, A2, B1) based on an internal risk assessment model. Grade A 
repr ...


In [8]:
import pandas as pd
import joblib
import shap

X_test = pd.read_csv("data/processed/X_test.csv")
calibrated_model = joblib.load("models/xgb_calibrated_model.pkl")

frozen_estimator = calibrated_model.calibrated_classifiers_[0].estimator
xgb_model_for_shap = frozen_estimator.estimator

explainer = shap.TreeExplainer(xgb_model_for_shap)

X_test_sample = X_test.iloc[:100]
shap_values = explainer.shap_values(X_test_sample)

print("Model and SHAP values ready!")

Model and SHAP values ready!


In [9]:
def build_retrieval_query(top_reasons_df):
    """
    Convert a top-reasons DataFrame (from get_top_reasons) into 
    a natural language query for ChromaDB retrieval.
    """
    feature_names = top_reasons_df['feature'].tolist()
    query = f"risk factors related to: {', '.join(feature_names)}"
    return query

# Test with our first applicant's top reasons (reusing our Week 4 function)
from explainability import get_top_reasons

applicant_reasons = get_top_reasons(applicant_index=0, X_data=X_test_sample, 
                                      shap_values_array=shap_values, top_n=3)
print("Applicant's top reasons:")
print(applicant_reasons)

query = build_retrieval_query(applicant_reasons)
print("\nGenerated query:", query)

Applicant's top reasons:
     feature   value  shap_contribution
1       term      60           0.322479
5  sub_grade      14           0.225355
0  loan_amnt  7450.0          -0.168386

Generated query: risk factors related to: term, sub_grade, loan_amnt


In [10]:
results = collection.query(
    query_texts=[query],
    n_results=3  # get top 3 relevant policy documents
)

print("Retrieved documents for this applicant:")
for doc_id, doc_text in zip(results['ids'][0], results['documents'][0]):
    print(f"\n--- {doc_id} ---")
    print(doc_text[:200], "...")

Retrieved documents for this applicant:

--- credit_grade ---
Credit Grade and Sub-Grade Policy

Loan applicants are assigned a credit grade (A through G) and a more granular 
sub-grade (e.g., A1, A2, B1) based on an internal risk assessment model. Grade A 
repr ...

--- loan_terms ---
Loan Term and Amount Policy

Loan term length (e.g., 36 months vs. 60 months) directly affects risk assessment. 
Longer loan terms carry greater uncertainty over the repayment period, including 
expos ...

--- debt_to_income ---
Debt-to-Income (DTI) Ratio Policy

The debt-to-income ratio measures an applicant's total monthly debt payments as 
a percentage of their gross monthly income. A high DTI ratio indicates that a 
signi ...


In [12]:
from dotenv import load_dotenv
import os
from groq import Groq

# Load the .env file so we can access our API key
load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
print("Key loaded:", api_key is not None)  # just confirms it loaded, doesn't print the actual key

# Create the Groq client
client = Groq(api_key=api_key)

# Make a simple test call
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "Say 'Hello, RAG pipeline is connected!' and nothing else."}
    ]
)

print(response.choices[0].message.content)

Key loaded: True
Hello, RAG pipeline is connected!


In [13]:
def generate_explanation(applicant_index, X_data, shap_values_array, collection, llm_client, top_n=3):
    """
    Full RAG pipeline: given an applicant, generate a plain-English, 
    policy-grounded explanation of their risk prediction.
    """
    # Step 1: Get top SHAP reasons for this applicant
    top_reasons = get_top_reasons(applicant_index, X_data, shap_values_array, top_n=top_n)
    
    # Step 2: Build a retrieval query and get relevant policy text
    feature_names = top_reasons['feature'].tolist()
    query = f"risk factors related to: {', '.join(feature_names)}"
    retrieval_results = collection.query(query_texts=[query], n_results=2)
    retrieved_context = "\n\n".join(retrieval_results['documents'][0])
    
    # Step 3: Build the prompt for the LLM
    reasons_text = "\n".join([
        f"- {row['feature']}: value = {row['value']}, impact = {row['shap_contribution']:.3f}"
        for _, row in top_reasons.iterrows()
    ])
    
    prompt = f"""You are a compliance assistant helping explain a credit decision to a loan applicant.

The applicant's top risk factors, based on our model's analysis, are:
{reasons_text}

Relevant underwriting policy context:
{retrieved_context}

Write a clear, plain-English explanation (3-4 sentences) for the applicant, explaining 
the main reasons behind their risk assessment. Ground your explanation in the policy 
context provided. Do not invent information not supported by the data or policy text 
above. Write in a professional, respectful tone."""

    # Step 4: Call the LLM
    response = llm_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}]
    )
    
    explanation = response.choices[0].message.content
    
    return {
        "top_reasons": top_reasons,
        "retrieved_context": retrieved_context,
        "explanation": explanation
    }

In [14]:
result = generate_explanation(
    applicant_index=0, 
    X_data=X_test_sample, 
    shap_values_array=shap_values, 
    collection=collection, 
    llm_client=client
)

print("=== TOP REASONS ===")
print(result['top_reasons'])
print("\n=== GENERATED EXPLANATION ===")
print(result['explanation'])

=== TOP REASONS ===
     feature   value  shap_contribution
1       term      60           0.322479
5  sub_grade      14           0.225355
0  loan_amnt  7450.0          -0.168386

=== GENERATED EXPLANATION ===
Your application was flagged for higher risk largely because of two policy‑related factors: the 60‑month loan term and the sub‑grade you were assigned. A longer term increases uncertainty in repayment, which our policy identifies as a higher risk factor. Your sub‑grade places you in one of the lower credit tiers (D, E, F, G), which our model shows are more likely to default. The loan amount of $7,450 actually lowers risk slightly, but the term and sub‑grade outweigh that benefit in the overall assessment.


In [15]:
# Recall our sub_grade encoding from Week 2 - let's verify what "14" means
sub_grades_sorted = sorted(['A1','A2','A3','A4','A5','B1','B2','B3','B4','B5',
                             'C1','C2','C3','C4','C5','D1','D2','D3','D4','D5',
                             'E1','E2','E3','E4','E5','F1','F2','F3','F4','F5',
                             'G1','G2','G3','G4','G5'])
print(sub_grades_sorted[14])

C5


In [16]:
# Recreate our encoding mappings from Week 2, so we can REVERSE them

sub_grades_sorted = sorted(['A1','A2','A3','A4','A5','B1','B2','B3','B4','B5',
                             'C1','C2','C3','C4','C5','D1','D2','D3','D4','D5',
                             'E1','E2','E3','E4','E5','F1','F2','F3','F4','F5',
                             'G1','G2','G3','G4','G5'])
sub_grade_decode = {i: sg for i, sg in enumerate(sub_grades_sorted)}

grade_decode = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F', 6: 'G'}

def decode_feature_value(feature_name, value):
    """Convert an encoded feature value back to a human-readable form."""
    if feature_name == 'sub_grade':
        return sub_grade_decode.get(int(value), value)
    elif feature_name == 'grade':
        return grade_decode.get(int(value), value)
    elif feature_name == 'term':
        return f"{int(value)} months"
    elif feature_name.startswith('home_ownership_') or feature_name.startswith('purpose_') or feature_name.startswith('verification_status_'):
        return "Yes" if value == 1 or value == True else "No"
    else:
        return value

# Quick test
print(decode_feature_value('sub_grade', 14))  # should print 'C5'
print(decode_feature_value('term', 60))        # should print '60 months'

C5
60 months


In [17]:
def generate_explanation(applicant_index, X_data, shap_values_array, collection, llm_client, top_n=3):
    """
    Full RAG pipeline: given an applicant, generate a plain-English, 
    policy-grounded explanation of their risk prediction.
    """
    # Step 1: Get top SHAP reasons for this applicant
    top_reasons = get_top_reasons(applicant_index, X_data, shap_values_array, top_n=top_n)
    
    # Step 1.5: Decode feature values to human-readable form
    top_reasons = top_reasons.copy()
    top_reasons['decoded_value'] = top_reasons.apply(
        lambda row: decode_feature_value(row['feature'], row['value']), axis=1
    )
    
    # Step 2: Build a retrieval query and get relevant policy text
    feature_names = top_reasons['feature'].tolist()
    query = f"risk factors related to: {', '.join(feature_names)}"
    retrieval_results = collection.query(query_texts=[query], n_results=2)
    retrieved_context = "\n\n".join(retrieval_results['documents'][0])
    
    # Step 3: Build the prompt using DECODED values, not raw encoded numbers
    reasons_text = "\n".join([
        f"- {row['feature']}: value = {row['decoded_value']}, impact = {row['shap_contribution']:.3f}"
        for _, row in top_reasons.iterrows()
    ])
    
    prompt = f"""You are a compliance assistant helping explain a credit decision to a loan applicant.

The applicant's top risk factors, based on our model's analysis, are:
{reasons_text}

Relevant underwriting policy context:
{retrieved_context}

Write a clear, plain-English explanation (3-4 sentences) for the applicant, explaining 
the main reasons behind their risk assessment. Ground your explanation strictly in the 
exact values and policy context provided above. Do not infer or state any specific 
category, tier, or range that is not explicitly given to you. Write in a professional, 
respectful tone."""

    # Step 4: Call the LLM
    response = llm_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}]
    )
    
    explanation = response.choices[0].message.content
    
    return {
        "top_reasons": top_reasons,
        "retrieved_context": retrieved_context,
        "explanation": explanation
    }

In [18]:
result = generate_explanation(
    applicant_index=0, 
    X_data=X_test_sample, 
    shap_values_array=shap_values, 
    collection=collection, 
    llm_client=client
)

print("=== TOP REASONS ===")
print(result['top_reasons'])
print("\n=== GENERATED EXPLANATION ===")
print(result['explanation'])

=== TOP REASONS ===
     feature   value  shap_contribution decoded_value
1       term      60           0.322479     60 months
5  sub_grade      14           0.225355            C5
0  loan_amnt  7450.0          -0.168386        7450.0

=== GENERATED EXPLANATION ===
Your application was evaluated with a 60‑month loan term, which our model identifies as a higher‑risk factor (impact 0.322). The sub‑grade of C5, which falls toward the lower end of the credit grading spectrum, added 0.225 to the overall risk score. The loan amount of $7,450 had a negative impact of –0.168, meaning it lowered risk because it represents a smaller financial commitment relative to your overall profile. These three elements together inform the risk assessment for your application.


In [19]:
for i in [5, 20, 50]:
    print(f"\n{'='*50}")
    print(f"APPLICANT {i}")
    print('='*50)
    
    result = generate_explanation(
        applicant_index=i, 
        X_data=X_test_sample, 
        shap_values_array=shap_values, 
        collection=collection, 
        llm_client=client
    )
    
    print(result['top_reasons'][['feature', 'decoded_value', 'shap_contribution']])
    print("\nExplanation:")
    print(result['explanation'])


APPLICANT 5
                 feature decoded_value  shap_contribution
5              sub_grade            C5           0.190516
24  acc_open_past_24mths           2.0          -0.173424
1                   term     36 months          -0.149643

Explanation:
Your application received a sub‑grade of **C5**, which adds a risk weight of **+0.191** to the assessment.  
In contrast, the fact that you opened only **2 new accounts** in the past 24 months provides a negative weight of **‑0.173**, indicating a lower risk signal.  
The chosen loan term of **36 months** contributes another negative weight of **‑0.150**, further reducing the overall risk estimate.  
These three factors together shape the final risk score used in our decision.

APPLICANT 20
      feature decoded_value  shap_contribution
8  addr_state      0.003631          -0.196779
5   sub_grade            D1           0.191740
1        term     36 months          -0.174697

Explanation:
Your application was evaluated using severa

In [21]:
def clean_emp_length(value):
    if pd.isna(value):
        return -1
    elif value == '< 1 year':
        return 0
    elif value == '10+ years':
        return 10
    else:
        return int(value.split()[0])

In [22]:
df_with_states = pd.read_csv("data/processed/loans_with_target.csv")

df_fair = df_with_states.drop(columns=['emp_title', 'title'])

df_fair['emp_length'] = df_fair['emp_length'].apply(clean_emp_length)
df_fair['term'] = df_fair['term'].str.strip().str.replace(' months', '', regex=False).astype(int)

group_b_columns = [
    'avg_cur_bal', 'mo_sin_old_rev_tl_op', 'tot_coll_amt', 'num_actv_bc_tl',
    'total_rev_hi_lim', 'tot_cur_bal', 'bc_util', 'bc_open_to_buy',
    'num_bc_sats', 'num_sats', 'total_bc_limit', 'mort_acc',
    'acc_open_past_24mths', 'total_bal_ex_mort'
]
for col in group_b_columns:
    df_fair[col] = df_fair[col].fillna(df_fair[col].median())

group_c_columns = [
    'pub_rec_bankruptcies', 'revol_util', 'dti', 'collections_12_mths_ex_med',
    'chargeoff_within_12_mths', 'tax_liens'
]
df_fair = df_fair.dropna(subset=group_c_columns)
df_fair = df_fair.dropna()

X_fair = df_fair.drop(columns=['default'])
y_fair = df_fair['default']

from sklearn.model_selection import train_test_split
X_train_fair, X_test_fair, y_train_fair, y_test_fair = train_test_split(
    X_fair, y_fair, test_size=0.2, random_state=42, stratify=y_fair
)

print("X_test_fair recreated, shape:", X_test_fair.shape)

X_test_fair recreated, shape: (269098, 46)


In [23]:
state_mapping = dict(zip(X_test['addr_state'].round(6), X_test_fair['addr_state']))
print(state_mapping.get(round(0.003631, 6)))

WV


In [24]:
def decode_feature_value(feature_name, value):
    """Convert an encoded feature value back to a human-readable form."""
    if feature_name == 'sub_grade':
        return sub_grade_decode.get(int(value), value)
    elif feature_name == 'grade':
        return grade_decode.get(int(value), value)
    elif feature_name == 'term':
        return f"{int(value)} months"
    elif feature_name == 'addr_state':
        return state_mapping.get(round(value, 6), value)
    elif feature_name.startswith('home_ownership_') or feature_name.startswith('purpose_') or feature_name.startswith('verification_status_'):
        return "Yes" if value == 1 or value == True else "No"
    else:
        return value

In [27]:
def generate_explanation(applicant_index, X_data, shap_values_array, collection, llm_client, model, top_n=3):
    # Get the actual prediction and probability for this applicant
    applicant_data = X_data.iloc[[applicant_index]]
    prediction = model.predict(applicant_data)[0]
    probability = model.predict_proba(applicant_data)[0][1]
    
    decision = "DENIED" if prediction == 1 else "APPROVED"
    
    top_reasons = get_top_reasons(applicant_index, X_data, shap_values_array, top_n=top_n)
    top_reasons = top_reasons.copy()
    top_reasons['decoded_value'] = top_reasons.apply(
        lambda row: decode_feature_value(row['feature'], row['value']), axis=1
    )
    top_reasons['direction'] = top_reasons['shap_contribution'].apply(
        lambda x: 'INCREASES risk' if x > 0 else 'DECREASES risk'
    )
    
    feature_names = top_reasons['feature'].tolist()
    query = f"risk factors related to: {', '.join(feature_names)}"
    retrieval_results = collection.query(query_texts=[query], n_results=2)
    retrieved_context = "\n\n".join(retrieval_results['documents'][0])
    
    reasons_text = "\n".join([
        f"- {row['feature']}: value = {row['decoded_value']}, impact = {row['shap_contribution']:.3f} ({row['direction']})"
        for _, row in top_reasons.iterrows()
    ])
    
    prompt = f"""You are a compliance assistant helping explain a credit decision to a loan applicant.

DECISION: The application was {decision}.
Predicted default risk probability: {probability:.1%}

The applicant's top risk factors, based on our model's analysis, are:
{reasons_text}

IMPORTANT: Each factor above is explicitly labeled as either "INCREASES risk" or 
"DECREASES risk". You MUST use this exact labeled direction for each factor.

Relevant underwriting policy context:
{retrieved_context}

Write a clear, plain-English explanation (4-5 sentences) for the applicant. Start by 
clearly stating the decision (approved or denied). Then explain the main factors behind 
the risk assessment, grounded strictly in the exact values and directions provided above. 
Do not infer or state any specific category, tier, or range not explicitly given. 
Write in a professional, respectful tone."""

    response = llm_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return {
        "decision": decision,
        "probability": probability,
        "top_reasons": top_reasons,
        "retrieved_context": retrieved_context,
        "explanation": response.choices[0].message.content
    }

In [28]:
result = generate_explanation(50, X_test_sample, shap_values, collection, client, calibrated_model)
print("Decision:", result['decision'])
print("Probability:", f"{result['probability']:.1%}")
print("\nExplanation:")
print(result['explanation'])

Decision: APPROVED
Probability: 19.2%

Explanation:
Your loan application has been approved.  
Our assessment indicates a predicted default risk probability of 19.2%.  
Three key factors influenced this risk score: a 60‑month term (INCREASES risk, impact 0.412), a sub‑grade of B2 (DECREASES risk, impact –0.378), and 11 active balance‑carrying lines of credit (INCREASES risk, impact 0.214).  
The longer loan term and the higher number of active lines add uncertainty, while the B2 sub‑grade reflects a more favorable credit profile.  
We will proceed with the approval, but we recommend monitoring your credit activity to maintain or improve your credit grade in the future.


In [29]:
def generate_explanation(applicant_index, X_data, shap_values_array, collection, llm_client, model, top_n=3):
    applicant_data = X_data.iloc[[applicant_index]]
    prediction = model.predict(applicant_data)[0]
    probability = model.predict_proba(applicant_data)[0][1]
    decision = "DENIED" if prediction == 1 else "APPROVED"
    
    top_reasons = get_top_reasons(applicant_index, X_data, shap_values_array, top_n=top_n)
    top_reasons = top_reasons.copy()
    top_reasons['decoded_value'] = top_reasons.apply(
        lambda row: decode_feature_value(row['feature'], row['value']), axis=1
    )
    top_reasons['direction'] = top_reasons['shap_contribution'].apply(
        lambda x: 'INCREASES risk' if x > 0 else 'DECREASES risk'
    )
    
    feature_names = top_reasons['feature'].tolist()
    query = f"risk factors related to: {', '.join(feature_names)}"
    retrieval_results = collection.query(query_texts=[query], n_results=2)
    
    # Build context WITH source labels attached to each document
    retrieved_docs = retrieval_results['ids'][0]
    retrieved_texts = retrieval_results['documents'][0]
    retrieved_context = "\n\n".join([
        f"[SOURCE: {doc_id}]\n{text}" 
        for doc_id, text in zip(retrieved_docs, retrieved_texts)
    ])
    
    reasons_text = "\n".join([
        f"- {row['feature']}: value = {row['decoded_value']}, impact = {row['shap_contribution']:.3f} ({row['direction']})"
        for _, row in top_reasons.iterrows()
    ])
    
    prompt = f"""You are a compliance assistant helping explain a credit decision to a loan applicant.

DECISION: The application was {decision}.
Predicted default risk probability: {probability:.1%}

The applicant's top risk factors, based on our model's analysis, are:
{reasons_text}

IMPORTANT: Each factor above is explicitly labeled as either "INCREASES risk" or 
"DECREASES risk". You MUST use this exact labeled direction for each factor.

Relevant underwriting policy context (each block is labeled with its source document):
{retrieved_context}

Write a clear, plain-English explanation (4-5 sentences) for the applicant. Start by 
clearly stating the decision. Then explain the main factors, grounded strictly in the 
exact values and directions provided above. When you reference a policy concept, add 
a citation in the format [Source: document_name] immediately after that sentence. 
Do not infer or state any specific category, tier, or range not explicitly given. 
Write in a professional, respectful tone."""

    response = llm_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return {
        "decision": decision,
        "probability": probability,
        "top_reasons": top_reasons,
        "retrieved_context": retrieved_context,
        "sources_used": retrieved_docs,
        "explanation": response.choices[0].message.content
    }

In [30]:
result = generate_explanation(50, X_test_sample, shap_values, collection, client, calibrated_model)
print("Decision:", result['decision'])
print("Sources used:", result['sources_used'])
print("\nExplanation:")
print(result['explanation'])

Decision: APPROVED
Sources used: ['credit_grade', 'loan_terms']

Explanation:
Your application was approved.  
The model estimates your default risk probability at 19.2%.  
Key factors that influenced this decision include the 60‑month loan term, which increases risk [Source: loan_terms], and the 11 active balance lines, which also increases risk.  
Conversely, your sub‑grade of B2 reduces risk, reflecting a strong credit profile [Source: credit_grade].  
Overall, the combination of these factors led to the approval of your loan.
